In [1]:
#Import pandas, matplotlib.pyplot, and seaborn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# the supplied CSV data file is the raw_data directory
df_2021 = pd.read_sas(r"../Raw data/LLCP2021.XPT", format="xport")

In [3]:
df_2021.head()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_FRTRES1,_VEGRES1,_FRUTSU1,_VEGESU1,_FRTLT1A,_VEGLT1A,_FRT16A,_VEG23A,_FRUITE1,_VEGETE1
0,1.0,1.0,b'01192021',b'01',b'19',b'2021',1100.0,b'2021000001',2.021000e+09,1.0,...,1.0,1.0,100.0,214.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79
1,1.0,1.0,b'01212021',b'01',b'21',b'2021',1100.0,b'2021000002',2.021000e+09,1.0,...,1.0,1.0,100.0,128.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79
2,1.0,1.0,b'01212021',b'01',b'21',b'2021',1100.0,b'2021000003',2.021000e+09,1.0,...,1.0,1.0,100.0,71.0,1.0,2.0,1.0,1.0,5.397605e-79,5.397605e-79
3,1.0,1.0,b'01172021',b'01',b'17',b'2021',1100.0,b'2021000004',2.021000e+09,1.0,...,1.0,1.0,114.0,165.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79
4,1.0,1.0,b'01152021',b'01',b'15',b'2021',1100.0,b'2021000005',2.021000e+09,1.0,...,1.0,1.0,100.0,258.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79


In [4]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438693 entries, 0 to 438692
Columns: 303 entries, _STATE to _VEGETE1
dtypes: float64(298), object(5)
memory usage: 1014.1+ MB


Below is a summary of all the variables I am keeping for training the model

Target variable:
1. _RFHYPE6 - Ever been told that blood pressure is high

Demograhics:
1. MARITAL - Marital status
2. _RACE - Race/ethnicity categories
3. _SEX - Respondent sex (constructed variable)
4. _AGEG5YR - Reported age in 5-year categories calculated variable
5. _AGE80 - Imputed age value collapsed above 80

Social determinants
1. _STATE - State Code
2. _METSTAT - Metropolitan Status
3. _URBSTAT - Urban/rural
4. EMPLOY1 - Employment status
5. _EDUCAG - Computed level of education completed categories
6. _INCOMG1 - Computed income categories
7. _HLTHPLN - Have any health insurance
8. PERSDOC3 - Have personal health care provider

Lifestyle and nutrition indicators 
1. _TOTINDA - Adults who reported doing physical activity or exercise in the past 30 days other than their regular job
2. _BMI5 - Computed BMI
3. _BMI5CAT - Computed BMI categories
4. _SMOKER3 - Computed smoking status
5. _CURECI1 - Current e-cig user calculated variable
6. DRNKANY5 - Computed adult having at least one drink of alcohol in the last 30 days
7. _FRUTSU1 - Total fruits consumed per day
8. _VEGESU1 - Total vegetables consumed per day
9. GRENDA1 - Computed dark green vegetables intake in times per day
10. FRNCHDA_ - Computed french fry intake in times per day|

In [6]:
#Keeping target variable + independent variables 
columns_to_keep = ["_STATE","PERSDOC3",	"_RFHYPE6",	"MARITAL",	"EMPLOY1",	"_METSTAT",	"_URBSTAT",	"_RFHLTH",	"_PHYS14D",	"_HLTHPLN",	"_TOTINDA",	"_RACE",	"_SEX",	"_AGEG5YR",	"_AGE80",	"_BMI5",	"_BMI5CAT",	"_EDUCAG",	"_INCOMG1",	"_SMOKER3",	"_CURECI1",	"DRNKANY5",	"_FRUTSU1",	"_VEGESU1", "GRENDA1_", "FRNCHDA_"]

In [7]:
#Creating a new dataframe to only store the required variables 
df_2021_n = df_2021[columns_to_keep].copy()

In [8]:
df_2021_n.head()

,_STATE,PERSDOC3,_RFHYPE6,MARITAL,EMPLOY1,_METSTAT,_URBSTAT,_RFHLTH,_PHYS14D,_HLTHPLN,...,_BMI5CAT,_EDUCAG,_INCOMG1,_SMOKER3,_CURECI1,DRNKANY5,_FRUTSU1,_VEGESU1,GRENDA1_,FRNCHDA_
0,1.0,1.0,1.0,1.0,7.0,1.0,1.0,2.0,3.0,1.0,...,1.0,2.0,3.0,3.0,1.0,2.0,100.0,214.0,5.700000e+01,4.300000e+01
1,1.0,2.0,2.0,9.0,8.0,1.0,1.0,1.0,1.0,1.0,...,NaN,4.0,9.0,4.0,1.0,2.0,100.0,128.0,1.400000e+01,5.397605e-79
2,1.0,2.0,2.0,3.0,7.0,1.0,1.0,1.0,1.0,1.0,...,3.0,2.0,2.0,4.0,1.0,2.0,100.0,71.0,5.397605e-79,1.400000e+01
3,1.0,1.0,2.0,1.0,7.0,1.0,1.0,1.0,1.0,1.0,...,4.0,2.0,5.0,4.0,1.0,1.0,114.0,165.0,1.000000e+01,5.700000e+01
4,1.0,1.0,1.0,1.0,8.0,2.0,1.0,2.0,3.0,1.0,...,3.0,1.0,2.0,4.0,1.0,2.0,100.0,258.0,1.000000e+02,2.900000e+01


In [9]:
df_2021_n.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438693 entries, 0 to 438692
Data columns (total 26 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _STATE    438693 non-null  float64
 1   PERSDOC3  438691 non-null  float64
 2   _RFHYPE6  438693 non-null  float64
 3   MARITAL   438688 non-null  float64
 4   EMPLOY1   435105 non-null  float64
 5   _METSTAT  431639 non-null  float64
 6   _URBSTAT  431639 non-null  float64
 7   _RFHLTH   438693 non-null  float64
 8   _PHYS14D  438693 non-null  float64
 9   _HLTHPLN  438693 non-null  float64
 10  _TOTINDA  438693 non-null  float64
 11  _RACE     438693 non-null  float64
 12  _SEX      438693 non-null  float64
 13  _AGEG5YR  438693 non-null  float64
 14  _AGE80    438693 non-null  float64
 15  _BMI5     391841 non-null  float64
 16  _BMI5CAT  391841 non-null  float64
 17  _EDUCAG   438693 non-null  float64
 18  _INCOMG1  438693 non-null  float64
 19  _SMOKER3  438693 non-null  float64
 20  _CUR

In [10]:
df_2021_n.describe().T

,count,mean,std,min,25%,50%,75%,max
_STATE,438693.0,30.742155,15.334888,1.000000e+00,20.0,31.0,41.0,78.0
PERSDOC3,438691.0,1.578870,0.892250,1.000000e+00,1.0,1.0,2.0,9.0
_RFHYPE6,438693.0,1.427244,0.699127,1.000000e+00,1.0,1.0,2.0,9.0
MARITAL,438688.0,2.397864,1.825151,1.000000e+00,1.0,1.0,3.0,9.0
EMPLOY1,435105.0,3.803519,2.873347,1.000000e+00,1.0,2.0,7.0,9.0
_METSTAT,431639.0,1.304113,0.460031,1.000000e+00,1.0,1.0,2.0,2.0
_URBSTAT,431639.0,1.144051,0.351142,1.000000e+00,1.0,1.0,1.0,2.0
_RFHLTH,438693.0,1.186985,0.547929,1.000000e+00,1.0,1.0,1.0,9.0
_PHYS14D,438693.0,1.611678,1.296879,1.000000e+00,1.0,1.0,2.0,9.0
_HLTHPLN,438693.0,1.370170,1.566496,1.000000e+00,1.0,1.0,1.0,9.0


In [11]:
df_2021_n.shape

(438693, 26)

The contnuous BMI variable has implied two decimal places. As per the summary stats above, min is 12, mean is around 28.5, and max is around 99. This suggests that we have some extreme BMI values that could be potentially outliers. 

A realistic BMI range, as per CDC BMI definition, extreme values (e.g. BMI <=12 and BMI>70) are typically considered bioligically implausible and may reflect measurement or reporting error in height or weight measurmements. As a result, these extreme values will be cleaned and converted to NaN 

In [13]:
#Import numpy
import numpy as np

# Step 1: Remove implausible values
df_2021_n['_BMI5'] = df_2021_n['_BMI5'].where(
    (df_2021_n['_BMI5'] >= 1200) & (df_2021_n['_BMI5'] <= 7000),
    np.nan
)

# Step 2: Check distribution
print(df_2021_n['_BMI5'].describe(percentiles=[0.95, 0.99]))

count    391602.000000
mean       2852.145132
std         643.142236
min        1200.000000
50%        2744.000000
95%        4062.000000
99%        4913.000000
max        6994.000000
Name: _BMI5, dtype: float64


The diet variables (_FRUTSU1,_VEGESU1, GRENDA1_, FRNCHDA_) are coded differently with implied two decimal places. There are two cleaning issues:

1. There are tiny artifact values that are very close to 0. These should be changed to 0 and not labelled as missing as NaN
2. There are very large values that seem unrealistic. For example, it is not plausible for one person to consume fruits 198 times in a day. Based on logic, I set thresholds for each variable where observations greater than the threshold will be converted to NaN 
- Fruit consumption - <=10
- Vegetabes - <=15
- Dark Green vegetables <=5
- French fries <=3

In [15]:
#Import numpy
import numpy as np

#Step 1: Convert tiny values to 0
tiny_threshold = 1e-10

nutrition_vars = ['_FRUTSU1', '_VEGESU1', 'GRENDA1_', 'FRNCHDA_']

for col in nutrition_vars:
    df_2021_n[col] = df_2021_n[col].mask(df_2021_n[col] < tiny_threshold, 0)

#Step 2: Apply domain thresholds
#Fruit
df_2021_n['_FRUTSU1'] = df_2021_n['_FRUTSU1'].where(df_2021_n['_FRUTSU1'] <= 1000, np.nan)

#Total vegetables
df_2021_n['_VEGESU1'] = df_2021_n['_VEGESU1'].where(df_2021_n['_VEGESU1'] <= 1500, np.nan)

#Dark green vegetables
df_2021_n['GRENDA1_'] = df_2021_n['GRENDA1_'].where(df_2021_n['GRENDA1_'] <= 500, np.nan)

#French fries
df_2021_n['FRNCHDA_'] = df_2021_n['FRNCHDA_'].where(df_2021_n['FRNCHDA_'] <= 300, np.nan)

#Step 3: Quick check
for col in nutrition_vars:
    print(f"\n{col} summary:")
    print(df_2021_n[col].describe(percentiles=[0.95, 0.99]))


_FRUTSU1 summary:
count    384717.000000
mean        132.807898
std         115.322893
min           0.000000
50%         100.000000
95%         329.000000
99%         571.000000
max        1000.000000
Name: _FRUTSU1, dtype: float64

_VEGESU1 summary:
count    374859.000000
mean        190.164384
std         136.220528
min           0.000000
50%         164.000000
95%         407.000000
99%         757.000000
max        1500.000000
Name: _VEGESU1, dtype: float64

GRENDA1_ summary:
count    391671.000000
mean         52.029083
std          54.916912
min           0.000000
50%          43.000000
95%         100.000000
99%         300.000000
max         500.000000
Name: GRENDA1_, dtype: float64

FRNCHDA_ summary:
count    392722.000000
mean         20.840202
std          27.715837
min           0.000000
50%          14.000000
95%          71.000000
99%         100.000000
max         300.000000
Name: FRNCHDA_, dtype: float64


In [16]:
#Target Variable: Have you ever been told that you have high blood pressure? 
df_2021_n['_RFHYPE6'].value_counts()

_RFHYPE6
1.0    264648
2.0    172133
9.0      1912
Name: count, dtype: int64

Note - 1 -No, 2 = Yes, and 9.0 = don't know/not sure/refused/missing. Drop all NA/Missing from the target variable 

In [18]:
#Keeping only non-missing observations for our target variable 
df_2021_n = df_2021_n[df_2021_n['_RFHYPE6']!= 9.0]

In [19]:
#Checking to make sure only missing observations were dropped
df_2021_n['_RFHYPE6'].value_counts()

_RFHYPE6
1.0    264648
2.0    172133
Name: count, dtype: int64

In [20]:
#Most of the variables have don't know/refused/missing special coded. These will need to replaced as NaN 
import numpy as np

cols = ['PERSDOC3','MARITAL','EMPLOY1','_RFHLTH','_PHYS14D','_HLTHPLN','_TOTINDA','_RFHYPE6','_RACE','_AGEG5YR','_EDUCAG','_INCOMG1','_SMOKER3','_CURECI1','DRNKANY5']
missing_codes = [9.0, 14.0]

df_2021_n[cols] = df_2021_n[cols].replace(missing_codes, np.nan)

In [21]:
df_2021_n.isna().sum()

_STATE          0
PERSDOC3      858
_RFHYPE6        0
MARITAL      4726
EMPLOY1      7998
_METSTAT     7030
_URBSTAT     7030
_RFHLTH      1075
_PHYS14D     9252
_HLTHPLN    16928
_TOTINDA      814
_RACE       10633
_SEX            0
_AGEG5YR    53371
_AGE80          0
_BMI5       46434
_BMI5CAT    46198
_EDUCAG      2290
_INCOMG1    93322
_SMOKER3    24639
_CURECI1    23666
DRNKANY5    26519
_FRUTSU1    53412
_VEGESU1    63189
GRENDA1_    46552
FRNCHDA_    45533
dtype: int64

In [22]:
#Count the number of missing values in each column and sort them.
missing = pd.concat([df_2021_n.isnull().sum(), 100 * df_2021_n.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending=True)

,count,%
_STATE,0,0.000000
_AGE80,0,0.000000
_SEX,0,0.000000
_RFHYPE6,0,0.000000
_TOTINDA,814,0.186363
PERSDOC3,858,0.196437
_RFHLTH,1075,0.246119
_EDUCAG,2290,0.524290
MARITAL,4726,1.082007
_URBSTAT,7030,1.609502


Note - I will have to employ different types of imputation for different variables: 

1. For variables with very low missing values (<2%), I will impute using the mode. Since it's a small %, it will be simpler to use the mode.
2. For variables with low-moderate missing (2-10%), I will inpute with mode AND create a missing indicator for each of the variables where 1=missing 0= non-missing
3. For variables with moderate missing (10-20%), I will impute using median AND create a missing indicator for each of the variables where 1=missing 0= non-missing
4. For high missing (>20%) - I will collapse into categories consisting of low (<35k), medium (35-100k), and high(>100k), and include a category for missing. INCOME is the only variable in this range and missing income values have a chance of being correlated with lower socioeconomic status, privacy concerns, vulnerability. As a result, missing is not random and could be a predictor of worse outcomes. Consequently, I did not impute for this variable and collapsed it into Low/Medium/High for better interpretability. 

In [24]:
#Variables with < 2% missing: _TOTINDA, PERSDOC3, _RFHLTH, _EDUCAG, MARITAL, _URBSTAT, _METSTAT, EMPLOY1
vars_2_missing = [
    '_TOTINDA', 'PERSDOC3', '_RFHLTH', '_EDUCAG',
    'MARITAL', '_URBSTAT', '_METSTAT', 'EMPLOY1'
]

for col in vars_2_missing:
    mode_value = df_2021_n[col].mode(dropna=True)[0]
    df_2021_n[col] = df_2021_n[col].fillna(mode_value)

print("Remaining missing (<2% vars):")
print(df_2021_n[vars_2_missing].isnull().sum())

Remaining missing (<2% vars):
_TOTINDA    0
PERSDOC3    0
_RFHLTH     0
_EDUCAG     0
MARITAL     0
_URBSTAT    0
_METSTAT    0
EMPLOY1     0
dtype: int64


In [25]:
#Variables with 2-10% missing: _PHYS14D , _RACE, _HLTHPLN, _CURECI1,  _SMOKER3, DRNKANY5
vars_2_10_missing = [
    '_PHYS14D', '_RACE', '_HLTHPLN',
    '_CURECI1', '_SMOKER3', 'DRNKANY5'
]

for col in vars_2_10_missing:
    # Missing indicator
    df_2021_n[col + '_missing'] = df_2021_n[col].isnull().astype(int)
    
    # Mode imputation
    mode_value = df_2021_n[col].mode(dropna=True)[0]
    df_2021_n[col] = df_2021_n[col].fillna(mode_value)

print("\nMissing indicator counts (2–10% vars):")
print(df_2021_n[[col + '_missing' for col in vars_2_10_missing]].sum())

print("\nRemaining missing (2–10% vars):")
print(df_2021_n[vars_2_10_missing].isnull().sum())


Missing indicator counts (2–10% vars):
_PHYS14D_missing     9252
_RACE_missing       10633
_HLTHPLN_missing    16928
_CURECI1_missing    23666
_SMOKER3_missing    24639
DRNKANY5_missing    26519
dtype: int64

Remaining missing (2–10% vars):
_PHYS14D    0
_RACE       0
_HLTHPLN    0
_CURECI1    0
_SMOKER3    0
DRNKANY5    0
dtype: int64


In [26]:
#Variables with 10-20% missing: GRENDA1_, FRNCHDA_, _BMI5, _FRUTSU1, _VEGESU1 
vars_10_20_missing = [
    'GRENDA1_', 'FRNCHDA_', '_FRUTSU1',
    '_VEGESU1', '_BMI5'
]

for col in vars_10_20_missing:
    # Missing indicator
    df_2021_n[col + '_missing'] = df_2021_n[col].isnull().astype(int)
    
    # Median imputation
    median_value = df_2021_n[col].median()
    df_2021_n[col] = df_2021_n[col].fillna(median_value)

print("\nMissing indicator counts (10–15% vars):")
print(df_2021_n[[col + '_missing' for col in vars_10_20_missing]].sum())

print("\nRemaining missing (10–15% vars):")
print(df_2021_n[vars_10_20_missing].isnull().sum())


Missing indicator counts (10–15% vars):
GRENDA1__missing    46552
FRNCHDA__missing    45533
_FRUTSU1_missing    53412
_VEGESU1_missing    63189
_BMI5_missing       46434
dtype: int64

Remaining missing (10–15% vars):
GRENDA1_    0
FRNCHDA_    0
_FRUTSU1    0
_VEGESU1    0
_BMI5       0
dtype: int64


In [27]:
#Variables with > 20% missing: _INCOMG1 

# Create new grouped income variable
df_2021_n['INCOME_GROUP'] = df_2021_n['_INCOMG1'].replace({
    1: 'Low', 2: 'Low', 3: 'Low',
    4: 'Medium', 5: 'Medium',
    6: 'High', 7: 'High',
    9: 'Missing'
})

# Optional: ensure any NaN is also labeled as 'Missing'
df_2021_n['INCOME_GROUP'] = df_2021_n['INCOME_GROUP'].fillna('Missing')

# Convert to category (optional but recommended)
df_2021_n['INCOME_GROUP'] = df_2021_n['INCOME_GROUP'].astype('category')

# Check distribution
print("\nIncome group distribution:")
print(df_2021_n['INCOME_GROUP'].value_counts(dropna=False))


Income group distribution:
INCOME_GROUP
Medium     155270
Low        101957
Missing     93322
High        86232
Name: count, dtype: int64


In [28]:
#Checking the Count the number of missing values in each column and sort them.
missing = pd.concat([df_2021_n.isnull().sum(), 100 * df_2021_n.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending=True)

,count,%
_STATE,0,0.000000
DRNKANY5,0,0.000000
_FRUTSU1,0,0.000000
_VEGESU1,0,0.000000
GRENDA1_,0,0.000000
FRNCHDA_,0,0.000000
_PHYS14D_missing,0,0.000000
_RACE_missing,0,0.000000
_HLTHPLN_missing,0,0.000000
_CURECI1_missing,0,0.000000


NOTE - BMI5CAT and _AGEG5YR will not be used in the training of the model as it has a lot of missing compared to _AGE80 and _BMI5 but can use it for EDA. 

In [30]:
df_2021_n.info()

<class 'pandas.core.frame.DataFrame'>
Index: 436781 entries, 0 to 438692
Data columns (total 38 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   _STATE            436781 non-null  float64 
 1   PERSDOC3          436781 non-null  float64 
 2   _RFHYPE6          436781 non-null  float64 
 3   MARITAL           436781 non-null  float64 
 4   EMPLOY1           436781 non-null  float64 
 5   _METSTAT          436781 non-null  float64 
 6   _URBSTAT          436781 non-null  float64 
 7   _RFHLTH           436781 non-null  float64 
 8   _PHYS14D          436781 non-null  float64 
 9   _HLTHPLN          436781 non-null  float64 
 10  _TOTINDA          436781 non-null  float64 
 11  _RACE             436781 non-null  float64 
 12  _SEX              436781 non-null  float64 
 13  _AGEG5YR          383410 non-null  float64 
 14  _AGE80            436781 non-null  float64 
 15  _BMI5             436781 non-null  float64 
 16  _BMI5CA

In [31]:
df_2021_n['_BMI5CAT'].value_counts()

_BMI5CAT
3.0    138325
4.0    130889
2.0    115111
1.0      6258
Name: count, dtype: int64

In [32]:
df_2021_n['_BMI5'].value_counts()

_BMI5
2744.0    49683
2663.0     4207
2746.0     3311
2441.0     3259
2712.0     2927
          ...  
6483.0        1
5182.0        1
6392.0        1
3926.0        1
5632.0        1
Name: count, Length: 3713, dtype: int64

Note: I currently have two variables, one continuous BMI variable and BMI Categories. BMI categories will be suitable for EDA but the continuous BMI is better for training the model as it will capture more nuanced changes. 

BMI Categories
1 = Underweight, 2 = Normal Weight, 3 = Overweight, 4 = Obese 

In [34]:
df_2021_n[['_FRUTSU1','_VEGESU1','GRENDA1_', 'FRNCHDA_']].describe().T

,count,mean,std,min,25%,50%,75%,max
_FRUTSU1,436781.0,128.800227,108.546506,0.0,58.0,100.0,200.0,1000.0
_VEGESU1,436781.0,186.396991,126.271100,0.0,123.0,164.0,215.0,1500.0
GRENDA1_,436781.0,51.087815,51.981462,0.0,14.0,43.0,71.0,500.0
FRNCHDA_,436781.0,20.128073,26.301559,0.0,7.0,14.0,29.0,300.0


Note - All the nutrition indicators are two implied decimal places 

In [36]:
#Creating a new target variable that is binary (0=No high NP, 1= High BP) 
df_2021_n['HIGHBP'] = df_2021_n['_RFHYPE6'].map({1: 0, 2: 1})

In [37]:
#Mean of BP Prevalance by age-group 
#Note - Group 14.0 is actually special code for missing/NA.
bp_age_means = df_2021_n.groupby('_AGEG5YR')[['HIGHBP']].mean()
bp_age_means

,HIGHBP
_AGEG5YR,
1.0,0.074813
2.0,0.107647
3.0,0.140707
4.0,0.181247
5.0,0.237019
6.0,0.296388
7.0,0.360210
8.0,0.424838
10.0,0.543119


In [38]:
#Mean of BP Prevalance by sex (1=Male, 2=Female)
bp_sex_means = df_2021_n.groupby('_SEX')[['HIGHBP']].mean()
bp_sex_means

,HIGHBP
_SEX,
1.0,0.414800
2.0,0.376151


In [39]:
#Mean of BP Prevalance by BMI Categories (1=Male, 2=Female)
bp_bmi_means = df_2021_n.groupby('_BMI5CAT')[['HIGHBP']].mean()
bp_bmi_means

,HIGHBP
_BMI5CAT,
1.0,0.249121
2.0,0.263632
3.0,0.393139
4.0,0.527195


In [40]:
#Mean of BP Prevalance by Income categories (1=Male, 2=Female)
bp_income_means = df_2021_n.groupby('_INCOMG1')[['HIGHBP']].mean()
bp_income_means

,HIGHBP
_INCOMG1,
1.0,0.491863
2.0,0.487642
3.0,0.436650
4.0,0.428246
5.0,0.378293
6.0,0.318133
7.0,0.264606


Note - As expected, we see increasing prevalance of high BP with older age, being male, being in the overweight/obese BMI Category, and also in the lower income levels. 

In [45]:
#Saving the current dataframe as a csv file
df_2021_n.to_csv('../data/brfss_2021_clean.csv', index=False)